In [1]:
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# =========================================================
# 1. Load datasets
# =========================================================
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

TEXT_COL = "text"
LABEL_COL = "label"

train_df = train_df[[TEXT_COL, LABEL_COL]].dropna().copy()
val_df   = val_df[[TEXT_COL, LABEL_COL]].dropna().copy()
test_df  = test_df[[TEXT_COL, LABEL_COL]].dropna().copy()

X_train_text = train_df[TEXT_COL].astype(str).values
X_val_text   = val_df[TEXT_COL].astype(str).values
X_test_text  = test_df[TEXT_COL].astype(str).values

y_train = train_df[LABEL_COL].astype(int).values
y_val   = val_df[LABEL_COL].astype(int).values
y_test  = test_df[LABEL_COL].astype(int).values


# =========================================================
# 2. TF-IDF representation
# =========================================================
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    max_features=20000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.98
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_val_tfidf   = vectorizer.transform(X_val_text)
X_test_tfidf  = vectorizer.transform(X_test_text)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF val shape:", X_val_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)


# =========================================================
# 3. Reduce dimension for Bayesian logistic regression
# =========================================================
# You can try 50, 100, 200
n_components = 100

svd = TruncatedSVD(n_components=n_components, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_val_svd   = svd.transform(X_val_tfidf)
X_test_svd  = svd.transform(X_test_tfidf)

# Optional but helpful: standardize dense SVD features
scaler = StandardScaler()
X_train_svd = scaler.fit_transform(X_train_svd)
X_val_svd   = scaler.transform(X_val_svd)
X_test_svd  = scaler.transform(X_test_svd)

# Add bias column
X_train = np.hstack([np.ones((X_train_svd.shape[0], 1)), X_train_svd])
X_val   = np.hstack([np.ones((X_val_svd.shape[0], 1)), X_val_svd])
X_test  = np.hstack([np.ones((X_test_svd.shape[0], 1)), X_test_svd])

print("Bayesian feature train shape:", X_train.shape)
print("Bayesian feature val shape:", X_val.shape)
print("Bayesian feature test shape:", X_test.shape)


# =========================================================
# 4. Bayesian logistic regression with Laplace approximation
# =========================================================
alpha = 1.0   # prior precision

def neg_log_posterior(w, X, y, alpha):
    z = X @ w
    p = expit(z)
    eps = 1e-12
    nll = -np.sum(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
    prior = 0.5 * alpha * np.sum(w ** 2)
    return nll + prior

def grad_neg_log_posterior(w, X, y, alpha):
    z = X @ w
    p = expit(z)
    return X.T @ (p - y) + alpha * w

def hessian_neg_log_posterior(w, X, alpha):
    z = X @ w
    p = expit(z)
    r = p * (1 - p)
    XR = X * r[:, None]
    H = X.T @ XR + alpha * np.eye(X.shape[1])
    return H

w0 = np.zeros(X_train.shape[1])

result = minimize(
    fun=neg_log_posterior,
    x0=w0,
    args=(X_train, y_train, alpha),
    jac=grad_neg_log_posterior,
    method="BFGS",
    options={"maxiter": 1000, "disp": True}
)

w_map = result.x
print("\nOptimization success:", result.success)
print("Final objective:", result.fun)

H = hessian_neg_log_posterior(w_map, X_train, alpha)
S_N = np.linalg.inv(H)

print("Posterior covariance shape:", S_N.shape)


# =========================================================
# 5. Bayesian predictive probability
# =========================================================
def predictive_proba_bayes(X, w_map, S_N):
    mu_a = X @ w_map
    sigma2_a = np.sum((X @ S_N) * X, axis=1)
    kappa = 1.0 / np.sqrt(1.0 + np.pi * sigma2_a / 8.0)
    return expit(kappa * mu_a)

train_probs = predictive_proba_bayes(X_train, w_map, S_N)
val_probs   = predictive_proba_bayes(X_val, w_map, S_N)
test_probs  = predictive_proba_bayes(X_test, w_map, S_N)

train_pred = (train_probs >= 0.5).astype(int)
val_pred   = (val_probs >= 0.5).astype(int)
test_pred  = (test_probs >= 0.5).astype(int)


# =========================================================
# 6. Evaluation
# =========================================================
print("\nTrain Results")
print("Accuracy:", accuracy_score(y_train, train_pred))
print(classification_report(y_train, train_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_train, train_pred))

print("\nValidation Results")
print("Accuracy:", accuracy_score(y_val, val_pred))
print(classification_report(y_val, val_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_pred))

print("\nTest Results")
print("Accuracy:", accuracy_score(y_test, test_pred))
print(classification_report(y_test, test_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_test, test_pred))

TF-IDF train shape: (4248, 20000)
TF-IDF val shape: (607, 20000)
TF-IDF test shape: (1214, 20000)
Bayesian feature train shape: (4248, 101)
Bayesian feature val shape: (607, 101)
Bayesian feature test shape: (1214, 101)
Optimization terminated successfully.
         Current function value: 79.415651
         Iterations: 234
         Function evaluations: 273
         Gradient evaluations: 273

Optimization success: True
Final objective: 79.41565122403644
Posterior covariance shape: (101, 101)

Train Results
Accuracy: 0.9988229755178908
              precision    recall  f1-score   support

           0     0.9990    0.9986    0.9988      2100
           1     0.9986    0.9991    0.9988      2148

    accuracy                         0.9988      4248
   macro avg     0.9988    0.9988    0.9988      4248
weighted avg     0.9988    0.9988    0.9988      4248

Confusion Matrix:
 [[2097    3]
 [   2 2146]]

Validation Results
Accuracy: 0.9950576606260296
              precision    recall  f